# Bonus PoC — fail-closed PII gate và idempotent replay

PoC này kiểm chứng cơ chế khó nhất trong `ARCHITECTURE.md`:

1. Batch chỉ được ghi sau khi mọi PII đã token hóa thành công.
2. Lỗi token service không tạo Delta table và không commit source offset.
3. Replay cùng batch dùng `MERGE` nên không nhân đôi `event_id`.
4. Silver chỉ giữ token có version và prompt đã redaction.

In [1]:
from __future__ import annotations

import hashlib
import hmac
import re
import shutil
from pathlib import Path

import polars as pl
from deltalake import DeltaTable, write_deltalake

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "requirements.txt").exists() and (p / "notebooks").exists()
)
TABLE = ROOT / "_lakehouse" / "scratch" / "bonus_pii_gate"
if TABLE.exists():
    shutil.rmtree(TABLE)

DEMO_KEY = b"local-lab-key-not-a-production-secret"
TOKEN_VERSION = "tok-v1"
EMAIL_RE = re.compile(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"\+?\d[\d -]{7,}\d")

In [2]:
events = [
    {"event_id": "e1", "source_offset": 10, "email": "a@example.com", "phone": "+84 901 111 111", "prompt": "Email a@example.com"},
    {"event_id": "e2", "source_offset": 11, "email": "b@example.com", "phone": "+84 902 222 222", "prompt": "Call +84 902 222 222"},
    {"event_id": "e3", "source_offset": 12, "email": "c@example.com", "phone": "+84 903 333 333", "prompt": "No PII in this sentence"},
    {"event_id": "e4", "source_offset": 13, "email": "d@example.com", "phone": "+84 904 444 444", "prompt": "Owner d@example.com"},
    {"event_id": "e5", "source_offset": 14, "email": "e@example.com", "phone": "+84 905 555 555", "prompt": "Call +84 905 555 555"},
    {"event_id": "e6", "source_offset": 15, "email": "a@example.com", "phone": "+84 906 666 666", "prompt": "Repeat identity a@example.com"},
    # Retry của e2: offset mới hơn phải thắng trước khi MERGE.
    {"event_id": "e2", "source_offset": 16, "email": "b@example.com", "phone": "+84 902 222 222", "prompt": "Retry for b@example.com"},
]
checkpoint = {"committed_offset": -1}
print(f"Input events: {len(events)}; unique event_id: {len({e['event_id'] for e in events})}")

Input events: 7; unique event_id: 6


In [3]:
class TokenServiceUnavailable(RuntimeError):
    pass


class TokenService:
    """Deterministic lab tokenizer with a controllable mid-batch failure."""

    def __init__(self, fail_after: int | None = None):
        self.fail_after = fail_after
        self.calls = 0

    def tokenize(self, value: str) -> str:
        if self.fail_after is not None and self.calls >= self.fail_after:
            raise TokenServiceUnavailable("simulated KMS/token service outage")
        self.calls += 1
        digest = hmac.new(DEMO_KEY, value.strip().lower().encode(), hashlib.sha256).hexdigest()[:16]
        return f"{TOKEN_VERSION}:{digest}"


def redact(text: str) -> str:
    return PHONE_RE.sub("[PHONE]", EMAIL_RE.sub("[EMAIL]", text))


def stage_batch(raw_events: list[dict], service: TokenService) -> pl.DataFrame:
    staged = []
    for event in raw_events:
        staged.append({
            "event_id": event["event_id"],
            "source_offset": event["source_offset"],
            "email_token": service.tokenize(event["email"]),
            "phone_token": service.tokenize(event["phone"]),
            "prompt_redacted": redact(event["prompt"]),
            "tokenization_version": TOKEN_VERSION,
        })
    # Retry cùng event_id: giữ source offset mới nhất.
    return pl.DataFrame(staged).sort("source_offset").unique("event_id", keep="last")


def apply_batch(batch: pl.DataFrame) -> dict:
    if not (TABLE / "_delta_log").exists():
        write_deltalake(str(TABLE), batch.to_arrow(), mode="overwrite")
        return {"operation": "initial write", "rows": batch.height}
    return (
        DeltaTable(str(TABLE))
        .merge(batch.to_arrow(), predicate="t.event_id = s.event_id", source_alias="s", target_alias="t")
        .when_matched_update_all()
        .when_not_matched_insert_all()
        .execute()
    )

## 1. Fault injection: token service hỏng giữa batch

`stage_batch` chỉ tạo dữ liệu trong memory. Writer và checkpoint chỉ chạy sau
khi hàm trả về đầy đủ, nên lỗi giữa batch không thể tạo partial Silver commit.

In [4]:
try:
    stage_batch(events, TokenService(fail_after=3))
    raise AssertionError("fault injection did not fail")
except TokenServiceUnavailable as exc:
    print("Expected failure:", exc)

fail_closed = not TABLE.exists() and checkpoint["committed_offset"] == -1
print("Table exists after failure:", TABLE.exists())
print("Committed offset after failure:", checkpoint["committed_offset"])
print("Fail-closed:", fail_closed)

Expected failure: simulated KMS/token service outage
Table exists after failure: False
Committed offset after failure: -1
Fail-closed: True


## 2. Khôi phục service, ghi batch và replay cùng offsets

In [5]:
staged = stage_batch(events, TokenService())
first_metrics = apply_batch(staged)
checkpoint["committed_offset"] = int(staged["source_offset"].max())
first_count = DeltaTable(str(TABLE)).count()

replayed = stage_batch(events, TokenService())
replay_metrics = apply_batch(replayed)
checkpoint["committed_offset"] = max(
    checkpoint["committed_offset"], int(replayed["source_offset"].max())
)
final = pl.from_arrow(DeltaTable(str(TABLE)).to_pyarrow_table()).sort("event_id")

print("First write:", first_metrics)
print("Replay MERGE:", replay_metrics)
print(f"Rows first/replay: {first_count} / {final.height}")
print("Committed offset:", checkpoint["committed_offset"])

First write: {'operation': 'initial write', 'rows': 6}
Replay MERGE: {'num_source_rows': 6, 'num_target_rows_inserted': 0, 'num_target_rows_updated': 6, 'num_target_rows_deleted': 0, 'num_target_rows_copied': 0, 'num_output_rows': 6, 'num_target_files_scanned': 1, 'num_target_files_skipped_during_scan': 0, 'num_target_files_added': 1, 'num_target_files_removed': 1, 'execution_time_ms': 22, 'scan_time_ms': 5, 'rewrite_time_ms': 0}
Rows first/replay: 6 / 6
Committed offset: 16


In [6]:
serialized = "\n".join(str(v) for row in final.iter_rows(named=True) for v in row.values())
e1_token = final.filter(pl.col("event_id") == "e1")["email_token"][0]
e6_token = final.filter(pl.col("event_id") == "e6")["email_token"][0]

checks = {
    "failure created no table or checkpoint": fail_closed,
    "retry deduplicated to six event IDs": final.height == 6 and final["event_id"].n_unique() == 6,
    "replay did not increase row count": first_count == final.height,
    "raw email absent from Silver": all(e["email"] not in serialized for e in events),
    "raw phone absent from Silver": all(e["phone"] not in serialized for e in events),
    "prompt PII was redacted": "[EMAIL]" in serialized and "[PHONE]" in serialized,
    "tokenization is deterministic": e1_token == e6_token,
    "token version is recorded": final["tokenization_version"].unique().to_list() == [TOKEN_VERSION],
    "replay is auditable as a new commit": len(DeltaTable(str(TABLE)).history()) >= 2,
}
for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
assert all(checks.values())
print("\nBonus PoC complete — all checks passed.")

[PASS] failure created no table or checkpoint
[PASS] retry deduplicated to six event IDs
[PASS] replay did not increase row count
[PASS] raw email absent from Silver
[PASS] raw phone absent from Silver
[PASS] prompt PII was redacted
[PASS] tokenization is deterministic
[PASS] token version is recorded
[PASS] replay is auditable as a new commit

Bonus PoC complete — all checks passed.


## Kết luận

PoC chứng minh atomic staging và idempotent replay ở quy mô nhỏ. Nó chưa thay
KMS/HSM thật, distributed transaction giữa Kafka offset và Delta commit, ACL,
key rotation hay load test. Đây là các giới hạn đã nêu trong architecture brief.